# Data Preprocessing: Preterm Birth Prediction

**Dataset:** CDC/NCHS 2025 Natality sample (10,000 records, 9 input features, 1 binary target)
**Goal:** Prepare the data for classification (Logistic Regression and Decision Tree) without data leakage.
**Order of work:** inspect → clean → split → impute → encode → scale → export

## 1. Loading the data

We load the 10,000-record sample and check its shape and column names. The expected shape is (10000, 10), the same as in the EDA report.

In [1]:
import pandas as pd
import numpy as np

# Load the 10,000-record sample
df = pd.read_csv("natality_sample.csv")

print(df.shape)          # expected: (10000, 10)
print(df.columns.tolist())
df.head()

(10000, 10)
['maternal_age_group', 'maternal_education', 'prior_live_births_now_living', 'prior_live_births_now_dead', 'prepregnancy_bmi', 'prepregnancy_diabetes', 'prepregnancy_hypertension', 'previous_preterm_birth', 'plurality', 'preterm_birth']


,maternal_age_group,maternal_education,prior_live_births_now_living,prior_live_births_now_dead,prepregnancy_bmi,prepregnancy_diabetes,prepregnancy_hypertension,previous_preterm_birth,plurality,preterm_birth
0,20_24,high_school_or_GED,1.0,0.0,26.6,no,no,no,singleton,0
1,25_29,high_school_or_GED,2.0,0.0,28.3,no,no,no,singleton,0
2,35_39,associate_degree,1.0,0.0,35.7,no,no,no,singleton,0
3,35_39,bachelor_degree,2.0,0.0,22.5,no,no,yes,singleton,0
4,35_39,doctorate_or_professional_degree,0.0,0.0,29.8,no,no,no,singleton,0


### Data types and missing values

We check the type of each column and count missing values per column. These counts should match the EDA report (1,152 missing cells in total).

In [2]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   maternal_age_group            10000 non-null  str    
 1   maternal_education            9859 non-null   str    
 2   prior_live_births_now_living  9923 non-null   float64
 3   prior_live_births_now_dead    9616 non-null   float64
 4   prepregnancy_bmi              9660 non-null   float64
 5   prepregnancy_diabetes         9930 non-null   str    
 6   prepregnancy_hypertension     9930 non-null   str    
 7   previous_preterm_birth        9930 non-null   str    
 8   plurality                     10000 non-null  str    
 9   preterm_birth                 10000 non-null  int64  
dtypes: float64(3), int64(1), str(6)
memory usage: 781.4 KB


maternal_age_group                0
maternal_education              141
prior_live_births_now_living     77
prior_live_births_now_dead      384
prepregnancy_bmi                340
prepregnancy_diabetes            70
prepregnancy_hypertension        70
previous_preterm_birth           70
plurality                         0
preterm_birth                     0
dtype: int64

### Target distribution

The target `preterm_birth` is 1 for a preterm birth and 0 for a term birth. We check the class balance, because it affects the split and the evaluation later.

In [3]:
df["preterm_birth"].value_counts(normalize=True)

preterm_birth
0    0.8938
1    0.1062
Name: proportion, dtype: float64

## 2. Inspecting the actual values

Encoding decisions must be based on the real values in the data, not assumptions. We treat two groups of columns differently:

- **Categorical columns** (stored as text): we list every category, including NaN, to find unexpected labels and rare categories.
- **Count columns** (numeric, stored as float only because of NaN): they stay numeric, but we list their distinct values to look for impossible or special codes.

In [4]:
categorical_cols = [
    "maternal_age_group", "maternal_education",
    "prepregnancy_diabetes", "prepregnancy_hypertension",
    "previous_preterm_birth", "plurality",
]
count_cols = ["prior_live_births_now_living", "prior_live_births_now_dead"]

print("##### CATEGORICAL COLUMNS #####")
for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False).sort_index())

print("\n##### COUNT COLUMNS (numeric) #####")
for col in count_cols:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False).sort_index())

##### CATEGORICAL COLUMNS #####

=== maternal_age_group ===
maternal_age_group
15             7
16            20
17            55
18            93
19           190
20_24       1967
25_29       2849
30_34       2789
35_39       1649
40_44        363
45_49         12
50_54          2
under_15       4
Name: count, dtype: int64

=== maternal_education ===
maternal_education
associate_degree                     739
bachelor_degree                     1869
doctorate_or_professional_degree     261
grade_8_or_less                       86
grade_9_to_12_no_diploma             715
high_school_or_GED                  3360
masters_degree                       722
some_college_no_degree              2107
NaN                                  141
Name: count, dtype: int64

=== prepregnancy_diabetes ===
prepregnancy_diabetes
no     9773
yes     157
NaN      70
Name: count, dtype: int64

=== prepregnancy_hypertension ===
prepregnancy_hypertension
no     9377
yes     553
NaN      70
Name: count, dtype: 

### Checking for invalid values

We look for impossible values: negative numbers, non-integer counts and unrealistic BMI values. The special code 9 in `prior_live_births_now_dead` is handled in Section 4.

In [5]:
for c in ["prior_live_births_now_living", "prior_live_births_now_dead"]:
    s = df[c].dropna()
    print(f"{c}: min={s.min()}, max={s.max()}, negatives={int((s < 0).sum())}, non-integers={int((s % 1 != 0).sum())}")

s = df["prepregnancy_bmi"].dropna()
print(f"prepregnancy_bmi: min={s.min()}, max={s.max()}, values <= 0: {int((s <= 0).sum())}")

prior_live_births_now_living: min=0.0, max=14.0, negatives=0, non-integers=0
prior_live_births_now_dead: min=0.0, max=9.0, negatives=0, non-integers=0
prepregnancy_bmi: min=13.1, max=68.7, values <= 0: 0


## 3. Separating features and target, then train/test split

**Why split now:** every learned preprocessing step (imputation, scaling, encoding) must be fitted on the training data only. If we fitted them on the full dataset, information from the test set would influence training, which is data leakage. So we split **before** any imputation, encoding or scaling.

**Choices:**
- `X` contains the 9 input features and `y` the target `preterm_birth`. The target is kept completely separate.
- **80% train / 20% test.**
- **Stratified split (`stratify=y`):** only about 10.6% of births are preterm. Stratification keeps that same proportion in both sets, so the test set contains enough preterm cases to evaluate recall and precision reliably.
- `random_state=42` makes the split reproducible.

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="preterm_birth")
y = df["preterm_birth"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

X_train: (8000, 9) | X_test: (2000, 9)
y_train: (8000,) | y_test: (2000,)


### Duplicate rows and overlap with the training set

Duplicates are kept because identical features can belong to different mothers. The overlap with the training set is a limitation of the evaluation.

In [7]:
dup_rows = df.duplicated().sum()
print(f"Exact duplicate rows: {dup_rows} ({dup_rows/len(df)*100:.2f}%)")   # expect 2,025

Exact duplicate rows: 2025 (20.25%)


In [8]:
cols = X_train.columns.tolist()

# One copy of each distinct feature row from the training set
train_unique = X_train.drop_duplicates().assign(_in_train=1)

# Left-join the test rows: _in_train is filled only when the same row exists in train
merged = X_test.merge(train_unique, on=cols, how="left")
assert len(merged) == len(X_test)   # one result row per test row

share = merged["_in_train"].notna().mean()
print(f"Test rows with an identical feature row in train: {share*100:.1f}%")

Test rows with an identical feature row in train: 33.8%


### Verifying the split

We check that the preterm proportion is preserved in both sets, and that the number of missing values is kept as-is (we have not imputed anything yet).

In [9]:
print("Preterm rate, train:", round(y_train.mean(), 4))
print("Preterm rate, test: ", round(y_test.mean(), 4))
print("Preterm cases, train:", y_train.sum(), "| test:", y_test.sum())
print("Missing values, train:", X_train.isna().sum().sum(), "| test:", X_test.isna().sum().sum())

Preterm rate, train: 0.1062
Preterm rate, test:  0.106
Preterm cases, train: 850 | test: 212
Missing values, train: 879 | test: 273


### Checking the rare plurality category

Triplets are very rare in this sample (6 records). We check how they ended up distributed across the two sets.

In [10]:
print("Train:", X_train["plurality"].value_counts().to_dict())
print("Test: ", X_test["plurality"].value_counts().to_dict())

Train: {'singleton': 7743, 'twins': 251, 'triplets': 6}
Test:  {'singleton': 1926, 'twins': 74}


### How are the missing values distributed?

We count the missing values per column in the training set and check whether the three yes/no columns go missing in the same rows. The comparison with the outcome follows in the next cell.

In [11]:
bin_cols = ["prepregnancy_diabetes", "prepregnancy_hypertension", "previous_preterm_birth"]

print("Missing values per column (train):")
print(X_train.isna().sum())

# Do the three yes/no columns go missing together?
m = X_train[bin_cols].isna()
print("\nRows missing all 3 binary columns:", m.all(axis=1).sum())
print("Rows missing at least one:", m.any(axis=1).sum())
print("Rows with any missing value:", X_train.isna().any(axis=1).sum(),
      f"({X_train.isna().any(axis=1).mean()*100:.2f}%)")

Missing values per column (train):
maternal_age_group                0
maternal_education              113
prior_live_births_now_living     57
prior_live_births_now_dead      300
prepregnancy_bmi                250
prepregnancy_diabetes            53
prepregnancy_hypertension        53
previous_preterm_birth           53
plurality                         0
dtype: int64

Rows missing all 3 binary columns: 53
Rows missing at least one: 53
Rows with any missing value: 647 (8.09%)


### Is the missingness related to the outcome?

For each column with missing values, we compare the preterm rate among records where it is missing and records where it is present. A large gap means the missingness itself is informative.

In [12]:
cols_with_na = X_train.columns[X_train.isna().any()].tolist()

for c in cols_with_na:
    miss = X_train[c].isna()
    print(f"{c:32s} missing: {y_train[miss].mean():.3f} (n={miss.sum()}) | present: {y_train[~miss].mean():.3f}")

maternal_education               missing: 0.168 (n=113) | present: 0.105
prior_live_births_now_living     missing: 0.053 (n=57) | present: 0.107
prior_live_births_now_dead       missing: 0.047 (n=300) | present: 0.109
prepregnancy_bmi                 missing: 0.076 (n=250) | present: 0.107
prepregnancy_diabetes            missing: 0.226 (n=53) | present: 0.105
prepregnancy_hypertension        missing: 0.226 (n=53) | present: 0.105
previous_preterm_birth           missing: 0.226 (n=53) | present: 0.105


## 4. Special code 9 in `prior_live_births_now_dead`
The EDA found one record with value 9. It may be a special code and not a real count, so until it is verified against the CDC codebook it is treated as missing and filled by the median imputer. This rule uses no statistics from the data, so it does not leak information.

In [13]:
TREAT_9_AS_MISSING = True   # set to False if the codebook says 9 is a real count

col = "prior_live_births_now_dead"
print("Rows with 9 -> train:", int((X_train[col] == 9).sum()), "| test:", int((X_test[col] == 9).sum()))

if TREAT_9_AS_MISSING:
    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train.loc[X_train[col] == 9, col] = np.nan
    X_test.loc[X_test[col] == 9, col] = np.nan

Rows with 9 -> train: 1 | test: 0



## 5. Handling Missing Numerical Values

### Objective
Handle missing values in numerical features to ensure that the dataset is complete before applying machine learning algorithms.

### Methodology
Median imputation is used to replace missing values in numerical columns.

The median is preferred over the mean because it is less sensitive to extreme values and outliers.

The following numerical features are considered:
- `prior_live_births_now_living`
- `prior_live_births_now_dead`
- `prepregnancy_bmi`

### Avoiding Data Leakage
To prevent data leakage:
1. The imputer is fitted only on the training set.
2. Missing values in the training set are replaced using the training medians.
3. The same calculated medians are applied to the test set.

This ensures that information from the test set does not influence the preprocessing process.


In [14]:
from sklearn.impute import SimpleImputer

# Numerical columns
numeric_cols = [
    "prior_live_births_now_living",
    "prior_live_births_now_dead",
    "prepregnancy_bmi"
]

# Create copies to preserve the original split
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Create and fit the imputer on training data
num_imputer = SimpleImputer(strategy="median")

X_train_clean[numeric_cols] = num_imputer.fit_transform(
    X_train[numeric_cols]
)

# Apply the same imputer to test data
X_test_clean[numeric_cols] = num_imputer.transform(
    X_test[numeric_cols]
)

print("Missing values in training data:")
print(X_train_clean[numeric_cols].isna().sum())

print("\nMissing values in test data:")
print(X_test_clean[numeric_cols].isna().sum())

Missing values in training data:
prior_live_births_now_living    0
prior_live_births_now_dead      0
prepregnancy_bmi                0
dtype: int64

Missing values in test data:
prior_live_births_now_living    0
prior_live_births_now_dead      0
prepregnancy_bmi                0
dtype: int64



## 6. Handling Missing Categorical Values

### Objective
Handle missing values in categorical features before applying machine learning algorithms.

### Methodology
Constant imputation is used to replace missing categorical values with the label `unknown`.

This approach is preferred to replacing missing values with the most frequent category because:
- It preserves the information that a value was missing.
- It avoids assuming that an unknown value belongs to the most frequent category.
- It is particularly useful for binary features such as diabetes, hypertension, and previous preterm birth.

### Avoiding Data Leakage
The imputer is fitted on the training set and then applied to both training and testing sets using the same strategy.


In [15]:

from sklearn.impute import SimpleImputer

# Categorical features
categorical_cols = [
    "maternal_age_group",
    "maternal_education",
    "prepregnancy_diabetes",
    "prepregnancy_hypertension",
    "previous_preterm_birth",
    "plurality"
]

# Create the categorical imputer
cat_imputer = SimpleImputer(
    strategy="constant",
    fill_value="unknown"
)

# Fit on training data and transform
X_train_clean[categorical_cols] = cat_imputer.fit_transform(
    X_train[categorical_cols]
)

# Apply the same imputer to testing data
X_test_clean[categorical_cols] = cat_imputer.transform(
    X_test[categorical_cols]
)

# Verify missing values
print("Training set:")
print(X_train_clean[categorical_cols].isna().sum())

print("\nTesting set:")
print(X_test_clean[categorical_cols].isna().sum())


Training set:
maternal_age_group           0
maternal_education           0
prepregnancy_diabetes        0
prepregnancy_hypertension    0
previous_preterm_birth       0
plurality                    0
dtype: int64

Testing set:
maternal_age_group           0
maternal_education           0
prepregnancy_diabetes        0
prepregnancy_hypertension    0
previous_preterm_birth       0
plurality                    0
dtype: int64



## 7. Encoding Categorical Variables

### Objective
Convert categorical features into numerical representations suitable for machine learning algorithms while preserving the nature of each variable.

### Methodology

Two encoding techniques are used:

**1. Ordinal Encoding**

Applied to:
- `maternal_age_group`
- `maternal_education`

These features have a natural order. Their categories are assigned numerical values according to their respective order.

**2. One-Hot Encoding**

Applied to:
- `prepregnancy_diabetes`
- `prepregnancy_hypertension`
- `previous_preterm_birth`
- `plurality`

These features are nominal and do not have a meaningful numerical order. One-Hot Encoding creates separate binary columns for their categories.

### Handling Unknown Values

Unknown maternal education values are identified using a separate indicator feature. The ordinal education value is imputed using the median calculated from the training set.

### Avoiding Data Leakage

All learned preprocessing operations are fitted using training data only. The fitted transformations are then applied to the testing data.


In [16]:

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

# 1. Define the natural order of maternal age categories
age_order = [
    "under_15", "15", "16", "17", "18", "19",
    "20_24", "25_29", "30_34", "35_39",
    "40_44", "45_49", "50_54"
]

age_mapping = {category: i for i, category in enumerate(age_order)}

# 2. Define the natural order of education categories
education_order = [
    "grade_8_or_less",
    "grade_9_to_12_no_diploma",
    "high_school_or_GED",
    "some_college_no_degree",
    "associate_degree",
    "bachelor_degree",
    "masters_degree",
    "doctorate_or_professional_degree"
]

education_mapping = {
    category: i + 1
    for i, category in enumerate(education_order)
}

# 3. Create copies of the datasets
X_train_encoded = X_train_clean.copy()
X_test_encoded = X_test_clean.copy()

# 4. Ordinal encoding: maternal age
X_train_encoded["maternal_age_group"] = (
    X_train_encoded["maternal_age_group"].map(age_mapping)
)

X_test_encoded["maternal_age_group"] = (
    X_test_encoded["maternal_age_group"].map(age_mapping)
)

# 5. Preserve unknown education information
for data in [X_train_encoded, X_test_encoded]:
    data["maternal_education_unknown"] = (
        data["maternal_education"] == "unknown"
    ).astype(int)

    data["maternal_education"] = (
        data["maternal_education"].map(education_mapping)
    )

# 6. Impute unknown ordinal education values
education_imputer = SimpleImputer(strategy="median")

X_train_encoded[["maternal_education"]] = (
    education_imputer.fit_transform(
        X_train_encoded[["maternal_education"]]
    )
)

X_test_encoded[["maternal_education"]] = (
    education_imputer.transform(
        X_test_encoded[["maternal_education"]]
    )
)

# 7. One-Hot Encoding for nominal features
nominal_cols = [
    "prepregnancy_diabetes",
    "prepregnancy_hypertension",
    "previous_preterm_birth",
    "plurality"
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_nominal = encoder.fit_transform(
    X_train_encoded[nominal_cols]
)

test_nominal = encoder.transform(
    X_test_encoded[nominal_cols]
)

encoded_cols = encoder.get_feature_names_out(nominal_cols)

train_nominal = pd.DataFrame(
    train_nominal,
    columns=encoded_cols,
    index=X_train_encoded.index
)

test_nominal = pd.DataFrame(
    test_nominal,
    columns=encoded_cols,
    index=X_test_encoded.index
)

# 8. Combine all numerical and encoded features
other_cols = numeric_cols + [
    "maternal_age_group",
    "maternal_education",
    "maternal_education_unknown"
]

X_train_final = pd.concat(
    [X_train_encoded[other_cols], train_nominal],
    axis=1
)

X_test_final = pd.concat(
    [X_test_encoded[other_cols], test_nominal],
    axis=1
)

print("Training set shape:", X_train_final.shape)
print("Testing set shape:", X_test_final.shape)

print("\nFinal feature names:")
print(X_train_final.columns.tolist())

print("\nMissing values:")
print("Train:", X_train_final.isna().sum().sum())
print("Test:", X_test_final.isna().sum().sum())


Training set shape: (8000, 18)
Testing set shape: (2000, 18)

Final feature names:
['prior_live_births_now_living', 'prior_live_births_now_dead', 'prepregnancy_bmi', 'maternal_age_group', 'maternal_education', 'maternal_education_unknown', 'prepregnancy_diabetes_no', 'prepregnancy_diabetes_unknown', 'prepregnancy_diabetes_yes', 'prepregnancy_hypertension_no', 'prepregnancy_hypertension_unknown', 'prepregnancy_hypertension_yes', 'previous_preterm_birth_no', 'previous_preterm_birth_unknown', 'previous_preterm_birth_yes', 'plurality_singleton', 'plurality_triplets', 'plurality_twins']

Missing values:
Train: 0
Test: 0


## 8. Checking BMI outliers

The EDA found 262 high BMI values (above 45.15). They are statistically unusual but not confirmed errors, and a BMI of 60 or more is rare yet plausible. **They are kept.** The IQR limits are computed on the **training set only**.

If Logistic Regression is affected, a `RobustScaler` or clipping at the training limits can be tested in the modeling stage.

In [17]:
bmi = X_train["prepregnancy_bmi"].dropna()
q1, q3 = bmi.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

for name, d in [("train", X_train), ("test", X_test)]:
    s = d["prepregnancy_bmi"].dropna()
    print(name, "BMI outliers:", int(((s < low) | (s > high)).sum()), "of", len(s))
print("Bounds computed on train:", round(low, 2), round(high, 2))

train BMI outliers: 202 of 7750
test BMI outliers: 60 of 1910
Bounds computed on train: 9.55 45.15



## 9. Feature Scaling

### Objective
Bring numerical features to a comparable scale to improve the performance and stability of scale-sensitive machine learning algorithms.

### Methodology
Standardization is performed using the StandardScaler technique.

For each feature, the transformation is calculated as:

Z = (X - μ) / σ

Where:
- X represents the original value.
- μ represents the mean calculated from the training set.
- σ represents the standard deviation calculated from the training set.
- Z represents the standardized value.

### Features to Scale
Standardization is applied to numerical and ordinal features:
- prior_live_births_now_living
- prior_live_births_now_dead
- prepregnancy_bmi
- maternal_age_group
- maternal_education

One-Hot Encoded binary features are kept unchanged.

### Avoiding Data Leakage
The scaler is fitted only on the training set. The same calculated mean and standard deviation are then applied to the testing set.

### Justification
Scaling is important for Logistic Regression because it is sensitive to differences in feature magnitudes. Decision Trees generally do not require feature scaling, but the standardized dataset can still be used for a consistent comparison between the models.


In [18]:

from sklearn.preprocessing import StandardScaler

# Features to scale
scale_cols = [
    "prior_live_births_now_living",
    "prior_live_births_now_dead",
    "prepregnancy_bmi",
    "maternal_age_group",
    "maternal_education"
]

# Create the scaler
scaler = StandardScaler()

# Fit on training data and transform
X_train_final[scale_cols] = scaler.fit_transform(
    X_train_final[scale_cols]
)

# Apply the same scaler to testing data
X_test_final[scale_cols] = scaler.transform(
    X_test_final[scale_cols]
)

# Verify the scaling
print("Training set means:")
print(X_train_final[scale_cols].mean().round(3))

print("\nTraining set standard deviations:")
print(X_train_final[scale_cols].std(ddof=0).round(3))

print("\nFinal shapes:")
print("Train:", X_train_final.shape)
print("Test:", X_test_final.shape)


Training set means:
prior_live_births_now_living    0.0
prior_live_births_now_dead     -0.0
prepregnancy_bmi               -0.0
maternal_age_group              0.0
maternal_education             -0.0
dtype: float64

Training set standard deviations:
prior_live_births_now_living    1.0
prior_live_births_now_dead      1.0
prepregnancy_bmi                1.0
maternal_age_group              1.0
maternal_education              1.0
dtype: float64

Final shapes:
Train: (8000, 18)
Test: (2000, 18)


## 10. Final Preprocessing Validation

### Objective
Verify the quality, consistency and structure of the processed training and testing datasets before model development.

### Validation criteria
1. **Missing values:** none remain.
2. **Data types:** all input features are numerical.
3. **Feature consistency:** train and test have identical columns in the same order.
4. **Duplicate features:** no duplicated columns.
5. **Target separation:** the target is not inside the input features.
6. **Data integrity:** all numerical values are finite.

In [19]:
print("1. MISSING VALUES")
print("Train:", X_train_final.isna().sum().sum())
print("Test:", X_test_final.isna().sum().sum())

print("\n2. DATA TYPES")
print("All numerical (train):", X_train_final.select_dtypes(include=np.number).shape[1] == X_train_final.shape[1])
print("All numerical (test):", X_test_final.select_dtypes(include=np.number).shape[1] == X_test_final.shape[1])

print("\n3. FEATURE CONSISTENCY")
print("Same columns and order:", X_train_final.columns.equals(X_test_final.columns))

print("\n4. DUPLICATE COLUMNS")
print("Train:", X_train_final.columns.duplicated().sum(), "| Test:", X_test_final.columns.duplicated().sum())

print("\n5. TARGET SEPARATION")
print("Target in X_train:", "preterm_birth" in X_train_final.columns)
print("Target in X_test:", "preterm_birth" in X_test_final.columns)

print("\n6. FINITE VALUES")
print("Train:", np.isfinite(X_train_final.to_numpy()).all())
print("Test:", np.isfinite(X_test_final.to_numpy()).all())

print("\n7. FINAL DIMENSIONS")
print("X_train:", X_train_final.shape, "| X_test:", X_test_final.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

1. MISSING VALUES
Train: 0
Test: 0

2. DATA TYPES
All numerical (train): True
All numerical (test): True

3. FEATURE CONSISTENCY
Same columns and order: True

4. DUPLICATE COLUMNS
Train: 0 | Test: 0

5. TARGET SEPARATION
Target in X_train: False
Target in X_test: False

6. FINITE VALUES
Train: True
Test: True

7. FINAL DIMENSIONS
X_train: (8000, 18) | X_test: (2000, 18)
y_train: (8000,) | y_test: (2000,)


## 11. Saving the processed datasets

The processed data is exported so that the model notebooks all use exactly the same split and the same transformations.

- `preprocessed_dataset.csv`: all processed rows with the target and a `split` column (`train` / `test`).
- `train_preprocessed.csv` and `test_preprocessed.csv`: the same data as separate files, target included.
- `preprocessing_objects.joblib`: the fitted imputers, encoder, scaler and mappings.

The target stays in its own column and was never used to fit any transformation.

In [20]:
import joblib

TARGET = "preterm_birth"

# Rows must still line up with the targets (both came from the same split)
assert X_train_final.index.equals(y_train.index)
assert X_test_final.index.equals(y_test.index)

train_out = X_train_final.copy()
train_out[TARGET] = y_train.values
test_out = X_test_final.copy()
test_out[TARGET] = y_test.values

full_out = pd.concat(
    [train_out.assign(split="train"), test_out.assign(split="test")],
    ignore_index=True
)

full_out.to_csv("preprocessed_dataset.csv", index=False)
train_out.to_csv("train_preprocessed.csv", index=False)
test_out.to_csv("test_preprocessed.csv", index=False)

joblib.dump({
    "num_imputer": num_imputer,
    "cat_imputer": cat_imputer,
    "education_imputer": education_imputer,
    "onehot_encoder": encoder,
    "scaler": scaler,
    "age_mapping": age_mapping,
    "education_mapping": education_mapping,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "nominal_cols": nominal_cols,
    "scale_cols": scale_cols,
    "final_columns": X_train_final.columns.tolist(),
    "treat_9_as_missing": TREAT_9_AS_MISSING,
}, "preprocessing_objects.joblib")

# Read the main file back to confirm it was written correctly
check = pd.read_csv("preprocessed_dataset.csv")
print("preprocessed_dataset.csv:", check.shape)
print(check["split"].value_counts().to_dict())
print("Preterm rate  train:", round(check.loc[check.split == "train", TARGET].mean(), 4),
      "| test:", round(check.loc[check.split == "test", TARGET].mean(), 4))
print("Missing values in file:", int(check.isna().sum().sum()))
check.head()

preprocessed_dataset.csv: (10000, 20)
{'train': 8000, 'test': 2000}
Preterm rate  train: 0.1062 | test: 0.106
Missing values in file: 0


,prior_live_births_now_living,prior_live_births_now_dead,prepregnancy_bmi,maternal_age_group,maternal_education,maternal_education_unknown,prepregnancy_diabetes_no,prepregnancy_diabetes_unknown,prepregnancy_diabetes_yes,prepregnancy_hypertension_no,prepregnancy_hypertension_unknown,prepregnancy_hypertension_yes,previous_preterm_birth_no,previous_preterm_birth_unknown,previous_preterm_birth_yes,plurality_singleton,plurality_triplets,plurality_twins,preterm_birth,split
0,-0.860204,-0.11077,-0.208590,-1.110047,-0.799103,0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0,train
1,-0.210993,-0.11077,0.822926,1.238845,1.092452,0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0,train
2,-0.210993,-0.11077,-0.571799,-1.110047,-0.168585,0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0,train
3,-0.860204,-0.11077,-0.935009,-0.327083,-0.168585,0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0,train
4,-0.860204,-0.11077,-1.152935,-1.110047,-1.429621,0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0,train


## 12. Summary of preprocessing decisions

This table summarizes every choice and its reason, for the preprocessing report.

In [21]:
decisions = pd.DataFrame([
    ("Train/test split", "80/20, stratified on preterm_birth, random_state=42, done before any fitting",
     "Keeps the ~10.6% preterm share in both sets and prevents data leakage."),
    ("maternal_age_group", "Ordinal encoding (13 ordered levels, 0 to 12), then standardized",
     "Natural order. Levels are not equally spaced (single years, then 5-year bands)."),
    ("maternal_education", "Ordinal encoding (8 ordered levels), unknown flagged in maternal_education_unknown, median imputed, standardized",
     "Natural order. Keeps the information that the value was missing (1.41%)."),
    ("prior_live_births_now_living", "Median imputation (training set), standardized",
     "Right-skewed count; the median is robust to extreme values."),
    ("prior_live_births_now_dead", "Value 9 treated as missing (configurable), median imputation, standardized",
     "Value 9 is unverified against the CDC codebook. 3.84% missing."),
    ("prepregnancy_bmi", "Median imputation, standardized, outliers kept (IQR limits computed on train)",
     "High-end outliers are statistically unusual but not confirmed errors."),
    ("Yes/no columns (diabetes, hypertension, previous preterm birth)", "Missing becomes an 'unknown' category, then one-hot encoded",
     "Missing is not treated as 'no'. Trade-off: three columns per feature instead of one."),
    ("plurality", "One-hot encoding (singleton, twins, triplets)",
     "Avoids implying equal spacing. Triplets are very rare (6 records)."),
    ("Duplicate rows", "Kept",
     "Identical features can belong to different mothers (EDA: 224 feature combinations have both outcomes). Overlap between train and test is measured in Section 3."),
    ("Class imbalance", "Not corrected here", "Handled in the modeling stage and only on the training set."),
    ("Data leakage", "All imputers, encoders and the scaler fitted on training data only", "The test set only goes through transform."),
], columns=["Item", "Treatment", "Reason"])

decisions.to_csv("preprocessing_decisions.csv", index=False)
decisions

,Item,Treatment,Reason
0,Train/test split,"80/20, stratified on preterm_birth, random_sta...",Keeps the ~10.6% preterm share in both sets an...
1,maternal_age_group,"Ordinal encoding (13 ordered levels, 0 to 12),...",Natural order. Levels are not equally spaced (...
2,maternal_education,"Ordinal encoding (8 ordered levels), unknown f...",Natural order. Keeps the information that the ...
3,prior_live_births_now_living,"Median imputation (training set), standardized",Right-skewed count; the median is robust to ex...
4,prior_live_births_now_dead,"Value 9 treated as missing (configurable), med...",Value 9 is unverified against the CDC codebook...
5,prepregnancy_bmi,"Median imputation, standardized, outliers kept...",High-end outliers are statistically unusual bu...
6,"Yes/no columns (diabetes, hypertension, previo...","Missing becomes an 'unknown' category, then on...",Missing is not treated as 'no'. Trade-off: thr...
7,plurality,"One-hot encoding (singleton, twins, triplets)",Avoids implying equal spacing. Triplets are ve...
8,Duplicate rows,Kept,Identical features can belong to different mot...
9,Class imbalance,Not corrected here,Handled in the modeling stage and only on the ...


## 13. Conclusion and notes for the next stage

The data is now clean, fully numeric, free of missing values and split into reproducible training and test sets, with every learned transformation fitted on the training set only.

**For the model notebooks**
- Load `train_preprocessed.csv` and `test_preprocessed.csv`. Do not refit any preprocessing step on the test data.
- Preterm births are only about 10.6% of the records, so accuracy alone is misleading. Report precision, recall, F1-score, ROC-AUC and the confusion matrix, and handle the imbalance (for example `class_weight="balanced"`) on the training set only.
- About a third of the test rows have an identical feature row in the training set (Section 3), so test scores may be slightly optimistic.
- In this split, all 6 triplet pregnancies are in the training set and none in the test set, so the models cannot be evaluated on triplets.
- The meaning of value 9 in `prior_live_births_now_dead` should be confirmed with the CDC codebook and, if needed, the flag in Section 4 changed.